# 02 - Build Raw Layer

Este notebook constrói a camada Raw a partir dos arquivos originais armazenados na Landing Zone.

## Objetivos

- Ler os arquivos Parquet da landing.
- Tratar diferenças de schema físico entre arquivos mensais.
- Selecionar as colunas necessárias para o case.
- Aplicar casts técnicos mínimos.
- Adicionar metadados de origem e processamento.
- Persistir os dados como tabela Delta no Unity Catalog.

A camada Raw não aplica regras de negócio ou filtros de qualidade avançados. Essas regras serão tratadas na camada Silver/Trusted.

#### 01. Libs Usadas

In [ ]:
import os
import sys
import importlib

#### 02. Import do projeto

In [ ]:
current_path = os.getcwd()

if current_path.endswith("/notebooks"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = current_path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for module_name in ["src.build_raw"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

import src.build_raw as raw

print("Raw table:", raw.RAW_TABLE_NAME)
print("Landing base path:", raw.LANDING_BASE_PATH)

from src.build_raw import create_raw

#### 03. Criar tabela Raw

In [1]:
YEAR = 2023
MONTHS = ["01", "02", "03", "04", "05"]

raw_df = create_raw(
    spark=spark,
    year=YEAR,
    months=MONTHS,
    mode="overwrite",
)

print(f"Raw records: {raw_df.count()}")

display(raw_df.limit(10))

NameError: name 'create_raw' is not defined

### 04. Validações da Raw

#### 4.1 Contagem por mês

In [ ]:
display(
    spark.table("workspace.ifood_case.raw_yellow_taxi_trips")
    .groupBy("source_year", "source_month")
    .count()
    .orderBy("source_year", "source_month")
)

#### 4.2 Schema

In [ ]:
spark.table("workspace.ifood_case.raw_yellow_taxi_trips").printSchema()

#### 4.3 Validação das colunas obrigatórias

In [ ]:
required_columns = {
    "VendorID",
    "passenger_count",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
}

raw_columns = set(
    spark.table("workspace.ifood_case.raw_yellow_taxi_trips").columns
)

missing_columns = required_columns - raw_columns

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Todas as colunas solicitadas estão na camada Bronze.")

#### 4.4 Confirmar tabela no catálogo

In [ ]:
display(
    spark.sql("SHOW TABLES IN workspace.ifood_case")
)